In [1]:
import pandas as pd
import numpy as np

# 데이터 불러오기
df = pd.read_csv("model_df_할인율NaN&음수처리.csv")

# '주차' 컬럼을 날짜(datetime)로 변환
df['주차'] = pd.to_datetime(df['주차'])

# 연도별 분리
df_2023 = df[df['기획년도'] == 2023].copy()
df_2024 = df[df['기획년도'] == 2024].copy()

# 공통/신규 카테고리 분리
cat_2023 = set(df_2023['카테고리'].unique())
cat_2024 = set(df_2024['카테고리'].unique())
common_cats = cat_2023 & cat_2024

df_2024_common = df_2024[df_2024['카테고리'].isin(common_cats)].copy()
df_2024_new = df_2024[~df_2024['카테고리'].isin(common_cats)].copy()

# 시즌별 목표 설정
season_target_rate = {'봄': 80, '여름': 80, '가을': 65, '겨울': 55, '사계절': 80}
season_target_roi = {'봄': 1.0, '여름': 1.5, '가을': 1.6, '겨울': 1.0, '사계절': 2.0}

# 정확한 보간을 위해 주차정렬
for df_part in [df_2023, df_2024_common, df_2024_new]:
    df_part.sort_values(by=['시즌', '카테고리', '주차'], inplace=True)

# 목표율 생성해놓고 초기화
df_2024_common['목표누적판매율_주차별'] = df_2024_common['ROI목표율_주차별'] = np.nan
df_2024_new['목표누적판매율_주차별'] = df_2024_new['ROI목표율_주차별'] = np.nan


## 23년도 24년도 모두 있는 카테고리

In [2]:
# 공통 카테고리 대상
for 시즌명 in df_2024_common['시즌'].unique():
    
    시즌_판매율_목표 = season_target_rate[시즌명]
    시즌_ROI_목표 = season_target_roi[시즌명]

    시즌_카테고리들 = df_2024_common[df_2024_common['시즌'] == 시즌명]['카테고리'].unique()

    for 카테고리 in 시즌_카테고리들:

        df_2024_카테고리 = df_2024_common[(df_2024_common['시즌'] == 시즌명) & (df_2024_common['카테고리'] == 카테고리)]
        주차_수 = df_2024_카테고리.shape[0]

        # 24년도 판매주차수(N)에 기반하여 23년도의 마지막 N주만 남겨두고 추이에 활용 -> 입고시기가 달라진 제품의 보간 오차 줄임
        판매율_추이 = df_2023[(df_2023['시즌'] == 시즌명) & (df_2023['카테고리'] == 카테고리)]['누적판매율'].values[-주차_수:]
        ROI_추이 = df_2023[(df_2023['시즌'] == 시즌명) & (df_2023['카테고리'] == 카테고리)]['ROI'].values[-주차_수:]

        if len(판매율_추이) == 0 or len(ROI_추이) == 0:
            continue

        # 판매율 목표: 선형 보간 + 로그 가중치 + 목표값 맞춤
        보간_판매율 = np.interp(np.linspace(0, len(판매율_추이)-1, 주차_수), np.arange(len(판매율_추이)), 판매율_추이)
        보간_판매율 = 보간_판매율 / 보간_판매율[-1] * 시즌_판매율_목표
        
        로그_가중치 = np.log(np.linspace(1, 주차_수, 주차_수))
        로그_가중치 = (로그_가중치 - 로그_가중치.min()) / (로그_가중치.max() - 로그_가중치.min())
        
        가중_판매율 = 보간_판매율 * 로그_가중치
        최종_판매율 = np.concatenate([[보간_판매율[0]], (가중_판매율[1:] / 가중_판매율[-1] * 시즌_판매율_목표)])

        # ROI 목표: 선형 보간 + 목표값 스케일링
        보간_ROI = np.interp(np.linspace(0, len(ROI_추이)-1, 주차_수), np.arange(len(ROI_추이)), ROI_추이)
        보간_ROI = 보간_ROI / 보간_ROI[-1] * 시즌_ROI_목표

        # 결과 입력
        df_2024_common.loc[(df_2024_common['시즌'] == 시즌명) & (df_2024_common['카테고리'] == 카테고리),
                           '목표누적판매율_주차별'] = np.round(최종_판매율, 4)

        df_2024_common.loc[(df_2024_common['시즌'] == 시즌명) & (df_2024_common['카테고리'] == 카테고리),
                           'ROI목표율_주차별'] = np.round(보간_ROI, 4)


## 24년도 신규 카테고리

In [3]:
for 시즌명 in df_2024_new['시즌'].unique():
    
    시즌_판매율_목표 = season_target_rate[시즌명]
    시즌_ROI_목표 = season_target_roi[시즌명]

    시즌_카테고리들 = df_2024_new[df_2024_new['시즌'] == 시즌명]['카테고리'].unique()

    # 2023년 시즌 전체 데이터에서 평균 추이 구하기
    시즌_기준_데이터 = df_2023[(df_2023['시즌'] == 시즌명) & (df_2023['총입고수량'] > 0)]
    
    시즌_평균_판매율_추이 = 시즌_기준_데이터.groupby('주차')['누적판매율'].mean().sort_index().values
    시즌_평균_ROI_추이 = 시즌_기준_데이터.groupby('주차')['ROI'].mean().sort_index().values

    for 카테고리 in 시즌_카테고리들:

        df_2024_카테고리 = df_2024_new[(df_2024_new['시즌'] == 시즌명) & (df_2024_new['카테고리'] == 카테고리)]
        주차_수 = df_2024_카테고리.shape[0]

        if len(시즌_평균_판매율_추이) == 0 or len(시즌_평균_ROI_추이) == 0:
            continue

        # 판매율 보간 + 로그 가중치 + 목표치 도달
        보간_판매율 = np.interp(np.linspace(0, len(시즌_평균_판매율_추이)-1, 주차_수),
                            np.arange(len(시즌_평균_판매율_추이)), 시즌_평균_판매율_추이)
        보간_판매율 = 보간_판매율 / 보간_판매율[-1] * 시즌_판매율_목표
        
        로그_가중치 = np.log(np.linspace(1, 주차_수, 주차_수))
        로그_가중치 = (로그_가중치 - 로그_가중치.min()) / (로그_가중치.max() - 로그_가중치.min())
        
        가중_판매율 = 보간_판매율 * 로그_가중치
        최종_판매율 = np.concatenate([[보간_판매율[0]], (가중_판매율[1:] / 가중_판매율[-1] * 시즌_판매율_목표)])

        # ROI 보간 + 스케일링
        보간_ROI = np.interp(np.linspace(0, len(시즌_평균_ROI_추이)-1, 주차_수), np.arange(len(시즌_평균_ROI_추이)), 시즌_평균_ROI_추이)
        보간_ROI = 보간_ROI / 보간_ROI[-1] * 시즌_ROI_목표

        # 결과 저장
        df_2024_new.loc[(df_2024_new['시즌'] == 시즌명) & (df_2024_new['카테고리'] == 카테고리),
                        '목표누적판매율_주차별'] = np.round(최종_판매율, 4)

        df_2024_new.loc[(df_2024_new['시즌'] == 시즌명) & (df_2024_new['카테고리'] == 카테고리),
                        'ROI목표율_주차별'] = np.round(보간_ROI, 4)


## 월별 목표율 생성

In [4]:
# 두 데이터프레임 합치기
df_2024_final = pd.concat([df_2024_common, df_2024_new], ignore_index=True)

# 컬럼 초기화
df_2024_final['목표누적판매율_월별'] = np.nan
df_2024_final['ROI목표율_월별'] = np.nan
df_2024_final['월별_누적판매율'] = np.nan
df_2024_final['월별_ROI_percent'] = np.nan

# 시즌 → 카테고리 → 월별 루프
for 시즌명 in df_2024_final['시즌'].unique():
    카테고리들 = df_2024_final[df_2024_final['시즌'] == 시즌명]['카테고리'].unique()

    for 카테고리 in 카테고리들:
        해당_데이터 = df_2024_final[
            (df_2024_final['시즌'] == 시즌명) & 
            (df_2024_final['카테고리'] == 카테고리)
        ]

        for 월 in 해당_데이터['월'].unique():
            월별_데이터 = 해당_데이터[해당_데이터['월'] == 월].sort_values(by='주차')

            if 월별_데이터.empty:
                continue

            # 월말 값 추출
            월말_판매율_목표 = 월별_데이터.iloc[-1]['목표누적판매율_주차별']
            월말_ROI_목표 = 월별_데이터.iloc[-1]['ROI목표율_주차별']
            월말_판매율_실제 = 월별_데이터.iloc[-1]['누적판매율']
            월말_ROI_실제 = 월별_데이터.iloc[-1]['ROI']*100

            # 월 전체에 적용
            조건 = (
                (df_2024_final['시즌'] == 시즌명) &
                (df_2024_final['카테고리'] == 카테고리) &
                (df_2024_final['월'] == 월)
            )
            df_2024_final.loc[조건, '목표누적판매율_월별'] = 월말_판매율_목표
            df_2024_final.loc[조건, 'ROI목표율_월별'] = 월말_ROI_목표
            df_2024_final.loc[조건, '월별_누적판매율'] = 월말_판매율_실제
            df_2024_final.loc[조건, '월별_ROI_percent'] = 월말_ROI_실제



## 백분율 변환 & 목표달성율 계산

In [5]:
# 실제 ROI → 백분율로 변환
df_2024_final['ROI_percent'] = df_2024_final['ROI'] * 100
# ROI 목표값도 백분율로 변환
df_2024_final['ROI목표율_주차별_percent'] = df_2024_final['ROI목표율_주차별'] * 100
df_2024_final['ROI목표율_월별_percent'] = df_2024_final['ROI목표율_월별'] * 100


# 목표 통합 지표 (판매율 + ROI 50:50)
df_2024_final['목표통합지표_주차별'] = 0.5 * df_2024_final['목표누적판매율_주차별'] + \
                                     0.5 * df_2024_final['ROI목표율_주차별_percent']

df_2024_final['목표통합지표_월별'] = 0.5 * df_2024_final['목표누적판매율_월별'] + \
                                    0.5 * df_2024_final['ROI목표율_월별_percent']

# 실제 통합 지표 (실제 판매율 + ROI)
df_2024_final['주차별_실제통합지표'] = 0.5 * df_2024_final['누적판매율'] + \
                                     0.5 * df_2024_final['ROI_percent']
                                     
df_2024_final['월별_실제통합지표'] = 0.5 * df_2024_final['월별_누적판매율'] + \
                                   0.5 * df_2024_final['월별_ROI_percent']


# 목표달성율 = (실제 / 목표) * 100
df_2024_final['통합목표달성율_주차별'] = (df_2024_final['주차별_실제통합지표'] / df_2024_final['목표통합지표_주차별']) *100
df_2024_final['통합목표달성율_월별'] = (df_2024_final['월별_실제통합지표'] / df_2024_final['목표통합지표_월별']) *100


# 소수점 처리
df_2024_final['목표누적판매율_주차별'] = df_2024_final['목표누적판매율_주차별'].round(2)
df_2024_final['목표누적판매율_월별'] = df_2024_final['목표누적판매율_월별'].round(2)
df_2024_final['ROI목표율_주차별'] = df_2024_final['ROI목표율_주차별'].round(2)
df_2024_final['ROI목표율_월별'] = df_2024_final['ROI목표율_월별'].round(2)
df_2024_final['통합목표달성율_주차별'] = df_2024_final['통합목표달성율_주차별'].round(2)
df_2024_final['통합목표달성율_월별'] = df_2024_final['통합목표달성율_월별'].round(2)


In [6]:
# 2023년용 목표/달성 관련 컬럼 NaN 생성
목표_관련_컬럼 = [
    '목표누적판매율_주차별', '목표누적판매율_월별',
    'ROI목표율_주차별', 'ROI목표율_월별',
    'ROI목표율_주차별_percent', 'ROI목표율_월별_percent',
    '목표통합지표_주차별', '목표통합지표_월별',
    '주차별_실제통합지표', '통합목표달성율_주차별', '통합목표달성율_월별'
]

for 컬럼명 in 목표_관련_컬럼:
    df_2023[컬럼명] = np.nan

# ROI만 % 변환
df_2023['ROI_percent'] = df_2023['ROI'] * 100

# 최종 병합
df_2324 = pd.concat([df_2023, df_2024_final], ignore_index=True)
df_2324 = df_2324.sort_values(by=['기획년도', '시즌', '카테고리', '주차']).reset_index(drop=True)


In [ ]:
# 겨울_사파리_패딩사파리_ZB 확인
# df_2324[(df_2324['시즌']=='겨울')&(df_2324['기획년도']==2024)]['카테고리'].unique()
df_2324[(df_2324['시즌']=='겨울')&(df_2324['기획년도']==2024)&(df_2324['카테고리'] == '겨울_사파리_패딩사파리_ZB')][['카테고리','주차','할인율','누적판매율','목표누적판매율_주차별','목표누적판매율_월별','ROI','ROI목표율_주차별','ROI목표율_월별','통합목표달성율_주차별','통합목표달성율_월별']].head(50)

,카테고리,주차,할인율,누적판매율,목표누적판매율_주차별,목표누적판매율_월별,ROI,ROI목표율_주차별,ROI목표율_월별,통합목표달성율_주차별,통합목표달성율_월별
2680,겨울_사파리_패딩사파리_ZB,2024-09-29,0.0,-0.05,0.38,0.38,-0.01,0.04,0.04,-24.39,-24.39
2681,겨울_사파리_패딩사파리_ZB,2024-10-06,20.0,-0.05,0.10,0.38,-0.01,0.04,0.04,-26.11,2.10
2682,겨울_사파리_패딩사파리_ZB,2024-10-13,20.0,-0.03,0.10,0.38,0.00,0.02,0.04,-1.46,2.10
2683,겨울_사파리_패딩사파리_ZB,2024-10-20,26.0,-0.02,0.20,0.38,0.00,0.02,0.04,-0.93,2.10
2684,겨울_사파리_패딩사파리_ZB,2024-10-27,33.0,0.09,0.38,0.38,0.00,0.04,0.04,2.10,2.10
2685,겨울_사파리_패딩사파리_ZB,2024-11-03,40.0,0.60,0.39,13.91,0.02,0.02,0.29,110.55,74.92
2686,겨울_사파리_패딩사파리_ZB,2024-11-10,51.0,1.60,1.27,13.91,0.05,0.08,0.29,72.47,74.92
2687,겨울_사파리_패딩사파리_ZB,2024-11-17,75.0,7.30,9.08,13.91,0.09,0.20,0.29,56.82,74.92
2688,겨울_사파리_패딩사파리_ZB,2024-11-24,74.0,15.46,13.91,13.91,0.17,0.29,0.29,74.92,74.92
2689,겨울_사파리_패딩사파리_ZB,2024-12-01,73.0,25.48,20.66,55.00,0.27,0.47,1.00,77.50,82.64


In [ ]:
# df_2324.to_csv("model_df(통합목표치추가).csv", index=False)